<center> <b> Build Architecture

> previous part we talk about custom dataset n dataloader, here we explore about `nn.Module` to undstand how model constructed in torch?

- `nn.Module` -> blueprint for all neur nets component
- all layer model in torch like linear,Conv2d, LayerNorm, TransformerEncoder it child from nn.Module
- we use nn.Module due to superpowers it brings, it allows torch automatically calculate trainable weights and determine which grads need to be calculated?

# Structure (basic)

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__() #<- running constuctor parent class nn.Module

    def forward(self,x):
        return x

## \__init__() & \__forward__()

In [ ]:
# init : layer definition
self.linear = nn.Linear(128,64)
# means: build layer -> set params -> params registration

In [ ]:
# forward: computational flow
x = self.linear(x)
# means: data passing the layer -> actual computation

In [3]:
# exmple again
class SimpleNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4,2)

    def forward(self,x):
        x = self.linear(x)
        return x

<style color='red'>vfdsf

# <font color='red'>  IMPORTANT - How to Reverse engineer paper to code?

most of the paper in ai field just combination from some of concept: matmul, linear, normalization, activation, attention etc.

e.g. we see $y = W_2\sigma(W_1x)$ <br>
we can unstand:
- $W_1x$ = Linear
- $\sigma$ = activation func
- $W_2$ = Linear also <br>
so in python we can write
    - Linear
        - Act func
        - Linear

In [ ]:
# code implementation
self.fc1 = nn.Linear(d,hidden)
self.actfunc = nn.GELU()
self.fc2 = nn.Linear(hidden,d)

> for instance the concept of FFN Transformer:

$FFN(x) = max(0,xW_1+b_1)W_2+b_2$

In [4]:
# in torch u can write as simple
class FFN(nn.Module):
    
    def __init__(self,d_model,hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(d_model,hidden_dim)
        self.leaky_relu = nn.LeakyReLU()
        self.fc2 = nn.Linear(hidden_dim,d_model)

    def forward(self,x):
        x = self.fc1(x)
        x = self.leaky_relu(x)
        x = self.fc2(x)

        return x

### Mapping paper notation 🤨
- $Wx+b$ `nn.Linear(in_dim,out_dim)`
- $\sigma(x)$ `nn.ReLu()` or whatever act func
- $LayerNorm(x)$ `nn.LayerNorm(d_model)`
- $x+f(x)$ `x = x+self.block(x)` <- residual connection hmm

In [3]:
# example of resnet block (👆🏻)
class ResNetBlock(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.fc = nn.Linear(dim,dim)

    def forward(self,x):
        out = self.fc(x)
        out = out + x
        return out

# Shape Tracking 101

## LM

let say we've sentences: "i love my mom so much" <br>
tokenizer => [101,1034,3214,2293, 2784, 4083,2022,102] => token_length = 8
- batch = 32
- max seq len = 128
- initial tensor become (B,T) = (32,128) -> tensor contains integer token ids (not yet embding)


Embedding layer<br>
`nn.Embedding(vocab_size,d_model)` -> e.g. `nn.Embedding(30000, 512)`
<br> each token is a 512 embedding dim


Shape after embedding phase<br>
input (32,128) -> out (32,128,512) (batch, token_cnt, embedding_dim)


> <b>Formula <br>

$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

### Attention

- input: (32,128,512)
- Transformers build vector: key, query, value
- QKV Projection
    ```
    self.q_proj = nn.Linear(512,512)
    self.k_proj = nn.Linear(512,512)
    self.v_proj = nn.Linear(512,512)
    ```
- out will same in Linear(512,512) so for each Q,K,V -> (32,128,512)

### Multihead

but.. remember that transformers have multihead architecture so we seperate the embedding dim into each of head
- embedding_dim = 512
- num_heads = 8 <br><br>
head_dim = $\frac{512}{8} = 64$ 
- reshape multihead => initial: (32,128,512) to (32,128,8,64) (B,T,n_head,emb_dim)
- why we need to seperate into many head? so that each head can learn such as..
    - syntax
    - name
    - semantic
    - long dependency
- transpose again (32,128,8,64) (B,T,n_head,emb_dim) -> (32,8,128,64) (B,n_head,T,emb_dim)


### Attention score

Formula: $QK^{T}$
- Q:(32,8,128,64)
- K:(32,8,128,64) -> $K^T$: (32,8,64,128) *transpose last dim
- so $((128,64)@(64,128)) = (128,128)$ *if u confused about the result, open again ur linalg book wkwk ⬇️
    - if 2 matrix with ordo (AxB) and (CxD) to be multiplied, it requires B=C! so the result is (row the 1st matrix) x (col the 2nd matrix) 
- full tensor become (128,128)
- final --> (32,8,128,128)

- softmax($QK^T$) -> it not change the shape
- multiply with $V$
    - (32,8,128,128)@(32,8,128,64) => (128,128)@(128,64) => (32,8,128,64)
- merge head back -> (transpose(1,2)&reshape) -> (32,128,8,64) -> (32,128,512)
- output projection -> `nn.Linear(512,512)` -> shape still (32,128,512) 
- attention block final -> (32,128,512) back in original form 😅

<font style='color: red'> <b >Mini Architecture

> Input: token ids => Output: contextual representation

In [ ]:
class MiniTransformers(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(30000,512)

        self.q_proj = nn.Linear(512,512)
        self.k_proj = nn.Linear(512,512)
        self.v_proj = nn.Linear(512,512)

        self.ffn1 = nn.Linear(512,2048)
        self.ffn2 = nn.Linear(2048,512)

        self.act = nn.GELU()

### Exercise

```
Token IDs
    ↓
Embedding
    ↓
QKV projection
    ↓
Attention-ish flow
    ↓
FFN
    ↓
Output
```

In [5]:
class SelfAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.q_proj = nn.Linear(16,16)
        self.k_proj = nn.Linear(16,16)
        self.v_proj = nn.Linear(16,16)

    def forward(self,x):
        print("\nINPUT")
        print(x.shape)

        # QKV PROJECTION
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        print("\nQ SHAPE")
        print(Q.shape)

        print("\nK SHAPE")
        print(K.shape)

        print("\nV SHAPE")
        print(V.shape)

        # ATTENTION SCORES
        scores = Q@K.transpose(-2,-1)
        
        print("\nSCORES SHAPE")
        print(scores.shape)

        # SCALE
        d_k = K.shape[-1]
        scores = scores / (d_k ** 0.5)

        # SOFTMAX
        weights = F.softmax(scores,dim=-1)

        print("\nATTENTION WEIGHTS")
        print(weights.shape)

        # WEIGHTS SUM
        out = weights @ V

        print("\nFINAL OUTPUT")
        print(out.shape)

        return out

In [3]:
x = torch.randn(2,5,16)

In [7]:
model = SelfAttention()
out = model(x)


INPUT
torch.Size([2, 5, 16])

Q SHAPE
torch.Size([2, 5, 16])

K SHAPE
torch.Size([2, 5, 16])

V SHAPE
torch.Size([2, 5, 16])

SCORES SHAPE
torch.Size([2, 5, 5])

ATTENTION WEIGHTS
torch.Size([2, 5, 5])

FINAL OUTPUT
torch.Size([2, 5, 16])


## CNN - Conv2D

- Input image: (B,C,H,W) -> (32,3,224,224)
- Layer will be:
    ```
    nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3)
- Conv make the size channel changed -> spatial layer will be smaller -> (32,3,222,222)
- 222 where does that come?
    - by default stride=1 and padding=0
    - formula: $H_{out} = \frac{H+2P-C}{s}+1$ = $\frac{224-3}{1}+1 = 222$
- Naa we've layer previously with (3,64,3) -> (in_channels, out_channels, kernel_size)
- conv layer simply change image into collection of feature map
    - 3 channel RGB -> 64 feature maps
- so the shape input:(32,3,222,222) -> layer:(3,64,3) -> out:(32,64,H',W')
    - h and w changed bcuz convultion
    - kernel always "eat" edge side(kernel size=3) -> filter 3x3
- padding add addtional border so (32,64,224,224)
- pooling make channel remain and smaller spatial

<b> CNN overall like this <br>
```
    Input
(32,3,224,224)
    ↓ Conv
(32,64,224,224)
    ↓ Pool
(32,64,112,112)
    ↓ Conv
(32,128,112,112)
    ↓ Pool
(32,128,56,56)
    ↓ Flatten
(32,128,7,7) *due to entered linear layer must be flatten
    ↓ x = torch.flatten(x,start_dim=1) (32,128*7*7)
(32,6272)
nn.Linear(6272,1000) *1000 its out class (ImageNet)
```

<font style='color: red'> <b >Mini Architecture

> conv: explore and learn about feature => pool: compress spatial => linear: classification reasoning

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        # (B,C,H,W) initial => (32,3,224,224)
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1) #in,out,kernel
        self.pool1 = nn.MaxPool2d(2)
        # B,C,H',W' => (32,32,112,112) 32 cuz learned 32 feature maps 112 affected by pool 224/2
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.pool2 = nn.MaxPool2d(2)
        # B,C,H',W' => (32,64,56,56)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)
        self.flatten = nn.Flatten()
        # B,C,H',W' => (32,128,28,28)
        self.fc = nn.Linear(128*28*28,10) 

    def forward(self,x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

## Encoder vs Decoder

### Encoder 🕹️

In [2]:
# dummy data
VOCAB = {
    '[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3,
    'The': 4, 'the': 5, 'cat': 6, 'sat': 7, 'on': 8,
    'mat': 9, 'dog': 10, 'ran': 11, 'fast': 12,
    'a': 13, 'big': 14, 'small': 15,
}

ID2WORD = {v:k for k,v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

# dummy sentences - 2 samples in one batch
# BERT FORMAT: [CLS] token1 token2 token3 ... [SEP]
# we mask certain position and model predict real token

In [3]:
# let say, real token: "The cat sat on the mat" and "The dog ran fast"
# after masking phase: "The dog [MASK] sat on the mat"  and "The dog [MASK] fast"
 
SENTENCES_MASKED = [
    # [CLS], The, [MASK], sat,  on,   the,  mat,  [SEP]
    [1,      4,   3,      7,    8,    5,    9,    2],
    # [CLS], The, dog,   [MASK], fast, [PAD], [PAD], [SEP]
    [1,      4,   10,    3,      12,   0,     0,     2],
]

SENTENCES_ORIGINAL = [
    [1, 4, 6,  7, 8, 5, 9,  2],   # label: 2nd position = cat (id=6)
    [1, 4, 10, 7, 12, 0, 0, 2],   # label: 3rd position = ran (id=11)
]

# Label: integer id from real token on position that masked
# -100 = ignored CrossEntropyLoss (non-mask position)
MASK_LABELS = [
    [-100, -100, 6,   -100, -100, -100, -100, -100],  # => "cat"
    [-100, -100, -100, 7,   -100, -100, -100, -100],  # => "ran"
]

In [4]:
input_ids  = torch.tensor(SENTENCES_MASKED)  # (B=2, T=8)
labels     = torch.tensor(MASK_LABELS)        # (B=2, T=8)
attn_mask  = (input_ids != VOCAB['[PAD]']).long()  # (B=2, T=8) — 0 di posisi PAD
 
print("=" * 60)
print("ENCODER (BERT-style MLM) FROM SCRATCH")
print("=" * 60)
print(f"\n[DATA]")
print(f"  input_ids   shape : {input_ids.shape}   dtype: {input_ids.dtype}")
print(f"  labels      shape : {labels.shape}   dtype: {labels.dtype}")
print(f"  attn_mask   shape : {attn_mask.shape}   dtype: {attn_mask.dtype}")
print(f"\n  Sentence 0 tokens : {[ID2WORD[i] for i in SENTENCES_MASKED[0]]}")
print(f"  Sentence 1 tokens : {[ID2WORD[i] for i in SENTENCES_MASKED[1]]}")

ENCODER (BERT-style MLM) FROM SCRATCH

[DATA]
  input_ids   shape : torch.Size([2, 8])   dtype: torch.int64
  labels      shape : torch.Size([2, 8])   dtype: torch.int64
  attn_mask   shape : torch.Size([2, 8])   dtype: torch.int64

  Sentence 0 tokens : ['[CLS]', 'The', '[MASK]', 'sat', 'on', 'the', 'mat', '[SEP]']
  Sentence 1 tokens : ['[CLS]', 'The', 'dog', '[MASK]', 'fast', '[PAD]', '[PAD]', '[SEP]']


#### Embedding

### Decoder ⚾︎